### External IP Ping Test

In [ ]:
!chcp 65001

In [ ]:
!ping 8.8.8.8

- 8.8.8.8은 Google의 공개 DNS 서버 IP이다.
- 8bytes를 송신하고 돌려받는 과정을 4번 반복한다.
- 근데 Received가 0이고, 모두 Lost되었다. = 100%는 패킷 손실률(Loss Rate)를 의미한다.

- Ping의 원리
    - 내 PC에서 ICMP Echo Request 패킷을 생성하여 목적지 IP로 송신
    - 목적지 서버가 패킷을 수신하면 ICMP Echo Reply 패킷을 다시 내 PC로 송신해준다.

##### 왜 차단되었을까?
1. 사내 / 학내 네트워크 또는 사설 와이파이의 ICMP 차단 [Config 단계]
2. Windows 방화벽 or 백신 프로그램의 차단 [소프트웨어 단계]
3. 가상 어댑터 간섭 [하드웨어 단계]
4. 실제 외부 인터넷 연결 차단 [물리적인 차단]

위의 4가지 중 적용되는 이유가 있는지 찾아보았다.

일단 나의 공유기(게이트웨이)로 ping을 보내본다.

In [ ]:
!ping 192.168.148.61

내 PC와 local network 간의 연결은 정상임을 알 수 있다.  


어디까지 가다가 막히는지 추적한다. 
- tracert는 경로 추적 라이브러리이다.

In [ ]:
!tracert -d 8.8.8.8

- 1번 게이트웨이부터 Request timed out이 걸린 것으로 보아, 내 PC 또는 공유기 단계의 차단임을 알 수 있다.
- 2~3번 게이트웨이부터 막힌다면, 상위 통신사(ISP) 또는 회사 방화벽 차단이라고 한다.

따라서, 인바운드 규칙을 일시적으로 허용한다.
1. Windows 키 -> Powershell 검색 후 관리자 권한으로 실행
2. ICMPv4(Ping) 일시 허용 명령어 입력: New-NetFirewallRule -Name "Allow_Ping" -DisplayName "ICMPv4-In (Allow Ping)" -Protocol ICMPv4 -IcmpType 8 -Action Allow -Enabled True
3. 실습 후 바로 차단: Remove-NetFirewallRule -Name "Allow_Ping"


In [ ]:
!ping 8.8.8.8

In [ ]:
!tracert -d 8.8.8.8

- ICMP을 임시로 해제했는데도 불가하다면, 그 상위의 회사 네트워크 단계에서 외부 핑을 차단했을 가능성이 높다.  
- 이 단계부턴 해결이 어려우므로, 진행하지 않았다.

### Internal IP Ping Test (Loopback)

In [ ]:
!ping 127.0.0.1

- 내 PC 내부의 루프백(loopback) IP 테스트 결과이다.